# Izbor atributa na osnovu modela stabla: HAR skup

**Pitanje:** koliko od 561 atributa može da se izbaci na osnovu važnosti iz modela stabla, a da klasifikator i dalje prepoznaje aktivnost skoro jednako dobro na test skupu?

**Skup:** Human Activity Recognition Using Smartphones (UCI). 30 osoba nosilo je telefon na pojasu. Iz signala akcelerometra i žiroskopa izračunat je 561 atribut. Cilj je jedna od 6 aktivnosti: hodanje, penjanje uz stepenice, silaženje niz stepenice, sedenje, stajanje, ležanje.

**Metrika:** accuracy i macro F1, jer su klase približno izbalansirane.

**Tolerancija:** smatramo da je rezultat „sličan“ ako je najviše 1 procentni poen (p.p.) ispod rezultata sa svim atributima.

## 0. Učitavanje podataka

Skup se skida sa UCI repozitorijuma (zip) u `data/`. Učitavaju se `X`, oznake aktivnosti i ID osobe za train i test deo.

In [1]:
from pathlib import Path
import io, zipfile, urllib.request
import pandas as pd

URL = "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip"
DATA = Path("../data/har")
ROOT = DATA / "UCI HAR Dataset"

# Skida se samo ako podaci već nisu lokalno (spoljni zip sadrži unutrašnji zip sa podacima)
if not ROOT.exists():
    DATA.mkdir(parents=True, exist_ok=True)
    outer = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(URL).read()))
    zipfile.ZipFile(io.BytesIO(outer.read("UCI HAR Dataset.zip"))).extractall(DATA)

# Imena atributa: neka se ponavljaju (bandsEnergy), pa im dodajemo redni broj
names = pd.read_csv(ROOT / "features.txt", sep=r"\s+", header=None)[1]
names = names.where(~names.duplicated(keep=False), names + "_" + names.index.astype(str))
activities = pd.read_csv(ROOT / "activity_labels.txt", sep=r"\s+", header=None, index_col=0)[1]

def load(part):
    X = pd.read_csv(ROOT / part / f"X_{part}.txt", sep=r"\s+", header=None)
    X.columns = names
    y = pd.read_csv(ROOT / part / f"y_{part}.txt", header=None)[0].map(activities)
    subject = pd.read_csv(ROOT / part / f"subject_{part}.txt", header=None)[0]
    return X, y, subject

X_train, y_train, subj_train = load("train")
X_test, y_test, subj_test = load("test")
print("train:", X_train.shape, "osoba:", subj_train.nunique())
print("test: ", X_test.shape, "osoba:", subj_test.nunique())

train: (7352, 561) osoba: 21
test:  (2947, 561) osoba: 9


## 1. Kratak pregled skupa

Broj primera po aktivnosti i broj osoba. Pokazujemo i koliko parova atributa ima visoku korelaciju (|r| > 0.9). To je dokaz da je dobar deo atributa redundantan, pa ima smisla izbacivati ih.

## 2. Podela podataka

Koristimo zvaničnu podelu **po osobama**: 21 osoba je u train, 9 u test skupu. Ista osoba nikad nije u oba skupa, pa test meri kako model radi na novim ljudima.

Test skup se ne dira do etape 7. Za izbor parametara koristi se cross-validation na train skupu, sa `GroupKFold` po osobi.

## 3. Baseline

- `DummyClassifier`: donja granica, uvek predviđa najčešću klasu.
- Random Forest na **svih 561 atributa**: ovo je rezultat sa kojim se porede svi manji skupovi atributa.

Merimo i vreme obučavanja i predikcije.

## 4. Važnost atributa

Random Forest se obučava na train skupu, pa iz njega računamo dve mere važnosti:

- **MDI** (`feature_importances_`): brza, ali pristrasna kod korelisanih atributa.
- **Permutaciona važnost**: koliko model gubi kada se vrednosti jednog atributa nasumično promešaju.

Prikazujemo top 20 atributa i poredimo rangiranja iz te dve mere.

## 5. Postepeno izbacivanje atributa

Za k = 561, 300, 150, 100, 50, 30, 20, 10, 5 zadržavamo top-k atributa i računamo CV rezultat istog Random Forest modela. Rangiranje se računa iznova unutar svakog folda, samo na njegovom train delu (`Pipeline` + `SelectFromModel`), da test fold ne bi uticao na izbor.

Na istom grafiku crtamo i **kontrolnu krivu**: k nasumično izabranih atributa. Ako je top-k jasno bolji od nasumičnog izbora, važnost iz modela zaista nešto donosi.

## 6. Izbor broja atributa

Biramo najmanji k čiji je CV rezultat u okviru tolerancije od baseline-a. Poredimo ga sa dva automatska načina:

- `SelectFromModel` sa pragom `mean` / `median`,
- `RFECV`: rekurzivno izbacuje najslabije atribute i sam bira k pomoću cross-validation.

## 7. Provera na test skupu

Model sa svim atributima i model sa izabranih k atributa obučavaju se na celom train skupu i **jednom** se ocenjuju na test skupu.

Prikazujemo accuracy, macro F1, matricu konfuzije i vreme obučavanja i predikcije. Očekujemo da se greške najviše javljaju između sedenja i stajanja.

## 8. Da li izbor važi i za druge klasifikatore?

Istih k atributa (izabranih pomoću Random Forest-a) dajemo **logističkoj regresiji** i **kNN-u**. Za svaki model poredimo rezultat sa svim atributima i sa k atributa.

Tako vidimo da li je izbor dobar uopšte ili samo za model stabla koji ga je napravio.

## 9. Stabilnost izbora

Koliko se top-k skupovi preklapaju između foldova i između više `random_state` vrednosti (Jaccard indeks). Ako se izabrani atributi mnogo menjaju, tumačenje konkretnih atributa nije pouzdano, čak i kad je rezultat stabilan.

## 10. Zaključak

Odgovor na početno pitanje treba da pokrije:

- koliko atributa je dovoljno i koliki je gubitak u odnosu na sve atribute;
- koliko se ubrzava obučavanje i predikcija;
- kada bismo preporučili ovaj pristup i kada ne;
- ograničenja eksperimenta: jedan skup, pristrasnost MDI važnosti kod korelisanih atributa, izbor tolerancije.